# Tarea 2 – Parte 2: Evaluación y extracción estructurada

**Muestra de evaluación (fija desde el inicio):** 10 documentos (3 impresos, 3 fotos y 4 manuscritos) y 10 clips (2 con TV de fondo, 3 de la conversación de trabajo y 5 de la familiar), tomados del corpus de la Parte 1. Las referencias son **transcripciones manuales**: las de los impresos y las fotos se copiaron del texto original (archivo de la tesis y PDF de SERVIR), y las de los manuscritos y los audios se tipearon a mano. En los manuscritos solo se transcribe lo escrito a mano (no el texto impreso de la agenda).

**Configuraciones definidas antes de calcular las métricas:** PaddleOCR y faster-whisper `small`, tal como en la Parte 1 (se reutilizan sus resultados); Tesseract 5.5 (`lang="spa"`, con la misma orientación que detectó PaddleOCR); Qwen3-ASR 0.6B (idioma español). Para el CER y el WER, la referencia y la transcripción se normalizan igual: minúsculas, sin tildes, sin puntuación y sin marcas `[HABLANTE_n]`.

In [5]:
import json
import re
import time
import unicodedata
from pathlib import Path

import torch  # Se importa antes que el resto, como en la Parte 1
import numpy as np
import pandas as pd
import jiwer
import pytesseract
from PIL import Image, ImageOps
from faster_whisper import decode_audio
from collections import Counter

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
SR = 16_000

In [2]:
# Muestra de evaluación: 10 documentos de los 3 tipos y 10 clips de las 3 grabaciones
MUESTRA_DOCS = ["doc_002", "doc_005", "doc_007", "doc_009", "doc_010", "doc_011",
                "doc_017", "doc_018", "doc_023", "doc_025"]
MUESTRA_CLIPS = ["clip_001", "clip_005", "clip_007", "clip_010", "clip_013",
                 "clip_015", "clip_017", "clip_019", "clip_024", "clip_025"]

documentos = pd.read_csv("datos/documentos.csv")
clips = pd.read_csv("datos/transcripciones.csv")
muestra_docs = documentos[documentos["id"].isin(MUESTRA_DOCS)].copy()
muestra_clips = clips[clips["id"].isin(MUESTRA_CLIPS)].copy()
print(muestra_docs["tipo"].value_counts().to_dict(), muestra_clips["fuente"].value_counts().to_dict())

{'manuscrito': 4, 'impreso': 3, 'foto': 3} {'Conversación familiar': 5, 'Conversación de trabajo': 3, 'Conversación con la hermana (TV de fondo)': 2}


In [3]:
# OCR alternativo: Tesseract 5.5 en español. Se le da la misma orientación que detectó el clasificador de
# PaddleOCR, porque su propio detector se equivocaba con las fotos giradas.
CARPETA_OCR = Path("datos/raw/documentos/ocr")
CARPETA_TESS = Path("datos/raw/documentos/tesseract")
CARPETA_TESS.mkdir(exist_ok=True)

def angulo_paddle(id_doc):
    """Ángulo (0, 90, 180 o 270) con que PaddleOCR enderezó el documento; 0 si no usó el clasificador."""
    with open(CARPETA_OCR / f"{id_doc}_res.json", encoding="utf-8") as f:
        return json.load(f).get("doc_preprocessor_res", {}).get("angle", 0)

textos, segundos = [], []
for fila in muestra_docs.itertuples():
    imagen = ImageOps.exif_transpose(Image.open(fila.archivo)).rotate(angulo_paddle(fila.id), expand=True)
    inicio = time.perf_counter()
    texto = pytesseract.image_to_string(imagen, lang="spa")
    segundos.append(round(time.perf_counter() - inicio, 1))
    textos.append(texto)
    (CARPETA_TESS / f"{fila.id}.txt").write_text(texto, encoding="utf-8")
    print(f"{fila.id}: {segundos[-1]} s")

muestra_docs["texto_tesseract"] = textos
muestra_docs["segundos_tesseract"] = segundos

doc_002: 3.7 s
doc_005: 1.9 s
doc_007: 2.8 s
doc_009: 6.9 s
doc_010: 17.2 s
doc_011: 11.0 s
doc_017: 7.8 s
doc_018: 6.3 s
doc_023: 7.7 s
doc_025: 7.8 s


In [4]:
# ASR alternativo: Qwen3-ASR 0.6B (Alibaba, Apache 2.0) en CPU, con el idioma fijado en español.
# Parakeet TDT 0.6B v3 se descartó: en una prueba previa mezclaba frases en inglés.
from transformers import AutoProcessor, Qwen3ASRForConditionalGeneration

CARPETA_QWEN = Path("datos/raw/audio/qwen3")
CARPETA_QWEN.mkdir(exist_ok=True)
proc_qwen = AutoProcessor.from_pretrained("Qwen/Qwen3-ASR-0.6B-hf")
modelo_qwen = Qwen3ASRForConditionalGeneration.from_pretrained("Qwen/Qwen3-ASR-0.6B-hf", dtype=torch.float32)

# Si un clip ya tiene su respaldo, no se vuelve a transcribir
for fila in muestra_clips.itertuples():
    respaldo = CARPETA_QWEN / f"{fila.id}.json"
    if respaldo.exists():
        print(f"{fila.id}: se lee el respaldo")
        continue
    audio = decode_audio(fila.archivo_audio, sampling_rate=SR)
    inicio = time.perf_counter()
    with torch.inference_mode():
        entrada = proc_qwen.apply_transcription_request(audio, language="es")
        salida = modelo_qwen.generate(**entrada, max_new_tokens=1024)
        texto = proc_qwen.decode(salida[0, entrada["input_ids"].shape[1]:], return_format="transcription_only").strip()
    segundos = time.perf_counter() - inicio
    respaldo.write_text(json.dumps({"id": fila.id, "texto": texto, "segundos_proceso": round(segundos, 1)},
                                   ensure_ascii=False), encoding="utf-8")
    print(f"{fila.id}: {fila.duracion_seg:.0f} s de audio en {segundos:.0f} s (RTF {segundos / fila.duracion_seg:.2f})")

Loading weights: 100%|██████████| 611/611 [00:27<00:00, 21.86it/s] 


clip_001: 54 s de audio en 110 s (RTF 2.05)
clip_005: 73 s de audio en 116 s (RTF 1.59)
clip_007: 75 s de audio en 80 s (RTF 1.06)
clip_010: 68 s de audio en 465 s (RTF 6.86)
clip_013: 59 s de audio en 65 s (RTF 1.11)
clip_015: 89 s de audio en 166 s (RTF 1.86)
clip_017: 76 s de audio en 113 s (RTF 1.48)
clip_019: 84 s de audio en 180 s (RTF 2.14)
clip_024: 82 s de audio en 172 s (RTF 2.10)
clip_025: 83 s de audio en 196 s (RTF 2.36)


## 2.2. Línea base sin ajustes

*Ejecutar la configuración base (PaddleOCR sin personalización adicional; faster-whisper small en CPU) sobre la muestra reservada y reportar CER (documentos) y WER (audio). Estaes la referencia contra la que compararán los modelos alternativos; no es obligatorio superarla.Normalicen las transcripciones de referencia y automáticas con el mismo criterio y, para WER,excluyan marcas como
[HABLANTE_1] que no sean palabras habladas.

In [6]:
# Normalización común para referencias y transcripciones: sin marcas de hablante, minúsculas, sin tildes ni puntuación
def normalizar(texto):
    texto = re.sub(r"\[HABLANTE_\d+\]", " ", texto).lower()
    texto = "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^\w\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

# Se comparan los textos crudos (sin seudonimizar), porque las referencias tienen los nombres tal como se oyen.
# Todo esto está en datos/raw/, que no se publica.
def leer_referencia(id_):
    return (Path("datos/raw/referencias") / f"{id_}.txt").read_text(encoding="utf-8")

def leer_json(ruta):
    with open(ruta, encoding="utf-8") as f:
        return json.load(f)

muestra_docs["referencia"] = muestra_docs["id"].apply(leer_referencia)
muestra_docs["texto_paddle"] = ["\n".join(leer_json(CARPETA_OCR / f"{i}_res.json")["rec_texts"]) for i in muestra_docs["id"]]

whisper = [leer_json(Path("datos/raw/audio/transcripciones") / f"{i}.json") for i in muestra_clips["id"]]
qwen = [leer_json(CARPETA_QWEN / f"{i}.json") for i in muestra_clips["id"]]
muestra_clips["referencia"] = muestra_clips["id"].apply(leer_referencia)
muestra_clips["texto_whisper"] = [" ".join(s["texto"] for s in w["segmentos"]) for w in whisper]
muestra_clips["texto_qwen"] = [q["texto"] for q in qwen]
muestra_clips["rtf_whisper"] = [w["segundos_proceso"] for w in whisper] / muestra_clips["duracion_seg"]
muestra_clips["rtf_qwen"] = [q["segundos_proceso"] for q in qwen] / muestra_clips["duracion_seg"]

In [7]:
def cer(referencia, hipotesis):
    return jiwer.cer(normalizar(referencia), normalizar(hipotesis) or "∅")

def wer(referencia, hipotesis):
    return jiwer.wer(normalizar(referencia), normalizar(hipotesis) or "∅")

def palabras_recuperadas(referencia, hipotesis):
    """Proporción de palabras de la referencia que aparecen en la transcripción, sin importar el orden."""
    ref, hip = Counter(normalizar(referencia).split()), Counter(normalizar(hipotesis).split())
    return sum((ref & hip).values()) / sum(ref.values())

for modelo in ["paddle", "tesseract"]:
    muestra_docs[f"cer_{modelo}"] = [cer(r, h) for r, h in zip(muestra_docs["referencia"], muestra_docs[f"texto_{modelo}"])]
    muestra_docs[f"recuperadas_{modelo}"] = [palabras_recuperadas(r, h) for r, h in zip(muestra_docs["referencia"], muestra_docs[f"texto_{modelo}"])]
for modelo in ["whisper", "qwen"]:
    muestra_clips[f"wer_{modelo}"] = [wer(r, h) for r, h in zip(muestra_clips["referencia"], muestra_clips[f"texto_{modelo}"])]

display(muestra_docs[["id", "tipo", "cer_paddle", "cer_tesseract", "recuperadas_paddle", "recuperadas_tesseract"]].round(3))
display(muestra_clips[["id", "fuente", "wer_whisper", "wer_qwen", "rtf_whisper", "rtf_qwen"]].round(2))

,id,tipo,cer_paddle,cer_tesseract,recuperadas_paddle,recuperadas_tesseract
1,doc_002,impreso,0.001,0.006,0.993,0.966
4,doc_005,impreso,0.000,0.002,1.000,0.994
6,doc_007,impreso,0.000,0.015,1.000,0.936
8,doc_009,foto,0.599,0.984,0.941,0.010
9,doc_010,foto,0.709,0.675,0.983,0.878
10,doc_011,foto,0.648,0.051,1.000,0.965
16,doc_017,manuscrito,0.177,0.913,0.456,0.018
17,doc_018,manuscrito,0.495,1.000,0.364,0.000
22,doc_023,manuscrito,0.118,0.754,0.667,0.039
24,doc_025,manuscrito,0.423,0.757,0.545,0.152


,id,fuente,wer_whisper,wer_qwen,rtf_whisper,rtf_qwen
0,clip_001,Conversación con la hermana (TV de fondo),0.23,0.25,1.01,2.05
4,clip_005,Conversación con la hermana (TV de fondo),0.42,0.44,1.15,1.59
6,clip_007,Conversación de trabajo,0.34,0.36,0.98,1.06
9,clip_010,Conversación de trabajo,0.57,0.52,0.85,6.86
12,clip_013,Conversación de trabajo,0.46,0.41,2.12,1.11
14,clip_015,Conversación familiar,0.32,0.27,1.06,1.86
16,clip_017,Conversación familiar,0.52,0.54,0.95,1.48
18,clip_019,Conversación familiar,0.47,0.48,1.04,2.14
23,clip_024,Conversación familiar,0.46,0.40,0.91,2.10
24,clip_025,Conversación familiar,0.42,0.34,0.84,2.36


In [8]:
# Métricas globales: jiwer suma los errores de los 10 casos y los divide entre el total de la referencia
def metrica_global(funcion, tabla, columna):
    return funcion(tabla["referencia"].apply(normalizar).tolist(),
                   tabla[columna].apply(normalizar).replace("", "∅").tolist())

CER_PADDLE = metrica_global(jiwer.cer, muestra_docs, "texto_paddle")
CER_TESSERACT = metrica_global(jiwer.cer, muestra_docs, "texto_tesseract")
WER_WHISPER = metrica_global(jiwer.wer, muestra_clips, "texto_whisper")
WER_QWEN = metrica_global(jiwer.wer, muestra_clips, "texto_qwen")
print(f"Documentos -> CER PaddleOCR (línea base): {CER_PADDLE:.3f} | CER Tesseract: {CER_TESSERACT:.3f}")
print(f"Audio -> WER faster-whisper small (línea base): {WER_WHISPER:.3f} (RTF {muestra_clips['rtf_whisper'].mean():.2f})"
      f" | WER Qwen3-ASR: {WER_QWEN:.3f} (RTF {muestra_clips['rtf_qwen'].mean():.2f})")

display(muestra_docs.groupby("tipo")[["cer_paddle", "cer_tesseract", "recuperadas_paddle", "recuperadas_tesseract"]].mean().round(3))
display(muestra_clips.groupby("fuente")[["wer_whisper", "wer_qwen", "rtf_whisper", "rtf_qwen"]].mean().round(2))

Documentos -> CER PaddleOCR (línea base): 0.402 | CER Tesseract: 0.500
Audio -> WER faster-whisper small (línea base): 0.423 (RTF 1.09) | WER Qwen3-ASR: 0.400 (RTF 2.26)


,cer_paddle,cer_tesseract,recuperadas_paddle,recuperadas_tesseract
tipo,,,,
foto,0.652,0.570,0.975,0.618
impreso,0.000,0.008,0.998,0.965
manuscrito,0.303,0.856,0.508,0.052


,wer_whisper,wer_qwen,rtf_whisper,rtf_qwen
fuente,,,,
Conversación con la hermana (TV de fondo),0.33,0.34,1.08,1.82
Conversación de trabajo,0.45,0.43,1.32,3.01
Conversación familiar,0.44,0.40,0.96,1.99
